# 🧠 Mini-Project 07: Disaggregated LLM Prefill/Decode Token Router & KV-Cache Optimizer
**PravahaQ (`v0.1.0`) — Standalone AI Infrastructure Showcase (Qiskit `BF-DCQO` + Deterministic Repair)**

---

## 🟢 Layer 1: Plain-English Intuition (Why Disaggregated LLM Serving Chokes)

Modern Large Language Model (LLM) serving engines (like vLLM, TensorRT-LLM, and Mooncake) split inference into two stages:
1. **Prefill Stage (Compute-Bound):** Processes the user's input prompt in parallel and generates a massive **Key-Value (KV) Cache tensor** (hundreds of megabytes per request).
2. **Decode Stage (Memory-Bandwidth-Bound):** Generates output tokens one by one, reading from that KV-Cache.

### The Bottleneck
If a **Prefill batch** runs on Server A and its coupled **Decode batch** runs on Server B across a slow PCIe/Ethernet link, transferring the KV-Cache stalls token generation (violating Time-To-First-Token SLAs). Conversely, packing both onto Server A while Server A is in Eco-Cooling mode causes **3-body thermal throttling**.

---

## 🔵 Layer 2: System Flowchart

```mermaid
flowchart LR
    A["Incoming LLM Stream\nPrefill + Decode Pairs"] --> B["KV-Cache Affinity Matrix W_mm'\n+ Inter-Node NVLink/RoCE Latency D_nn'"]
    B --> C["Vaisheshika 3-Body HUBO\nLatency + KV Sync + Thermal Triad"]
    C --> D["Qiskit v2 BF-DCQO Solver\n(4 Counterdiabatic Impulses)"]
    D --> E["100% SLA-Valid GPU Routing Table"]
```

---

## 🟣 Layer 3: Mathematical Formulation

Let $x_{m,n} \in \{0, 1\}$ indicate whether LLM stage $m \in \{1, \dots, M\}$ executes on GPU node $n \in \{1, \dots, N_{\text{gpu}}\}$, and $y_{n,r} \in \{0, 1\}$ denote cooling/power mode $r$. We minimize:
$$E_{\text{LLM}}(\mathbf{x}, \mathbf{y}) = \sum_{m,n} C^{\text{TTFT}}_{m,n} x_{m,n} + \sum_{m < m'} \sum_{n, n'} W^{\text{KV}}_{m,m'} D_{n,n'} x_{m,n} x_{m',n'} + \sum_{n, m < m'} T_{m,m',n} x_{m,n} x_{m',n} y_{n,\text{eco}} + \Phi_{\text{unbal}}(\mathbf{x})$$

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd() / "src"))

from pravaha_q.domain import generate_coupled_problem
from pravaha_q.math_engine import VaisheshikaHUBOCompiler, hubo_to_ising, FeasibilityRepairEngine
from pravaha_q.solvers import QiskitBFDCQOSolver

# 1. Synthesize a 3-Node (Jetson Orin, L40S, MI300A), 4-Stage LLM Prefill/Decode Cluster
llm_problem = generate_coupled_problem(
    num_nodes=3,
    num_tasks=4,
    num_power_states=1,
    seed=77,
    name="Disaggregated-LLM-KV-Router",
)

# 2. Compile to 3-Local Vaisheshika HUBO & Ising Spin Glass
hubo = VaisheshikaHUBOCompiler(enable_cubic_terms=True).compile(llm_problem)
ising = hubo_to_ising(hubo)

# 3. Execute Qiskit v2 BF-DCQO Solver & Decode Guaranteed-Feasible Schedule
qiskit_res = QiskitBFDCQOSolver(shots=2048, seed=77).solve(ising)
schedule = FeasibilityRepairEngine(llm_problem, hubo).repair_and_decode(qiskit_res.best_bitstring)

print("=== Mini-Project 07: Disaggregated LLM Prefill/Decode Cluster Router ===")
print(f"Cluster Topology          : {llm_problem.num_nodes} Heterogeneous GPU Nodes | {llm_problem.num_tasks} LLM Stages (2 Prefill + 2 Decode)")
print(f"Total Binary Variables    : {llm_problem.total_binary_variables} ({llm_problem.num_routing_vars} Routing + {llm_problem.num_power_vars} Cooling States)")
print(f"Qiskit BF-DCQO Energy     : {schedule.repaired_energy:.4f} | Execution Latency: {qiskit_res.execution_time_ms:.2f} ms")
print(f"SLA Feasibility Verified? : {schedule.is_feasible} (Violations Fixed: {schedule.constraint_violations_fixed})\n")

print("LLM Stage        | Assigned Accelerator Node       | Thermal Load | Power Mode")
print("-" * 82)
for m, task in enumerate(llm_problem.workload.tasks):
    node_idx = schedule.task_to_node[m]
    node_obj = llm_problem.cluster.nodes[node_idx]
    p_idx = schedule.node_to_power_state[node_idx]
    p_label = llm_problem.microgrid.power_states[p_idx].label
    print(f"{task.name:16s} | {node_obj.name:31s} |   {task.thermal_load_watts:6.2f} W   | {p_label}")

=== Mini-Project 07: Disaggregated LLM Prefill/Decode Cluster Router ===
Cluster Topology          : 3 Heterogeneous GPU Nodes | 4 LLM Stages (2 Prefill + 2 Decode)
Total Binary Variables    : 15 (12 Routing + 3 Cooling States)
Qiskit BF-DCQO Energy     : -39.4210 | Execution Latency: 11.45 ms
SLA Feasibility Verified? : True (Violations Fixed: 0)

LLM Stage        | Assigned Accelerator Node       | Thermal Load | Power Mode
----------------------------------------------------------------------------------
LLM-Batch-0      | EdgeNode-1-L40S                 |    68.40 W   | Solar_BESS_Eco
LLM-Batch-1      | EdgeNode-1-L40S                 |    52.10 W   | Solar_BESS_Eco
LLM-Batch-2      | EdgeNode-2-MI300A               |    88.90 W   | Solar_BESS_Eco
LLM-Batch-3      | EdgeNode-0-Jetson               |    49.25 W   | Solar_BESS_Eco
